# Capa Silver - Dimensión de Productos

In [1]:
from spark_session import *
from lakehouse_utils import *
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Spark Session

In [2]:
spark = create_spark_session(
    "retail-dlh-gold",
    configs=[
        configure_iceberg_delta_extensions,
        configure_delta_catalog,
        configure_polaris_restcatalog,
        configure_retail_polaris,
        configure_s3
    ]
)

## Extracción

In [3]:
target_table = "polaris.gold.DIM_PRODUCTS"
df_products = spark.table("silver.products")
df_dim_products = spark.table(target_table)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformaciones

In [4]:
# Obtener valor maximo de la llave subrogada
max_sk = df_dim_products.select(f.coalesce(f.max(f.col("PRODUCT_SK")), f.lit(0))).first()[0]

# Obtener registros nuevos y calcula su llave subrogada
df_products_ins = df_products.alias("p") \
    .join(df_dim_products.alias("dp"), (f.col("p.PRODUCT_ID") == f.col("dp.PRODUCT_ID")), "left_anti") \
    .withColumn("PRODUCT_SK", max_sk + f.row_number().over(Window.orderBy(f.col("PRODUCT_ID")))) \
    .select(
        f.col("PRODUCT_SK"),
        f.col("p.*")
    )

# Obtener registros a actualizar
df_products_upd = df_products.alias("p") \
    .join(df_dim_products.alias("dp"), (f.col("p.PRODUCT_ID") == f.col("dp.PRODUCT_ID")), "inner") \
    .select(
        f.col("dp.PRODUCT_SK"),
        f.col("p.*")
    )

# Seleccion de columnas finales
df_source = df_products_ins.unionByName(df_products_upd)

## Carga

In [5]:
# Obtenemos variables para merge
join_columns = ["PRODUCT_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)

## Cierre de SparkSession

In [6]:
spark.stop()